# Data I/O — counts, records, DataFrames

Build an `ObservationalData` object from contingency tables, row lists, or pandas.


In [ ]:
from pathlib import Path
from partialid import Assumptions, Query, bound, from_counts, from_records

roots = [Path.cwd(), Path.cwd().parent]
if "__file__" in globals():
    roots.insert(0, Path(__file__).resolve().parent.parent)
CSV = next((r / "examples" / "data" / "xy_counts.csv" for r in roots if (r / "examples" / "data" / "xy_counts.csv").exists()), Path("examples/data/xy_counts.csv"))
print("CSV:", CSV.resolve() if CSV.exists() else CSV)
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)


## From aggregated counts


In [ ]:
data = from_counts(
    ("X", "Y"),
    {(0, 0): 30, (0, 1): 20, (1, 0): 15, (1, 1): 35},
)
print(data.notes)
print("P:", {k: round(v, 4) for k, v in sorted(data.P.items())})
r = bound(data.variables, data.P, Q, Assumptions.manski("X", "Y"))
print(r.summary(Q))


## From individual records (list of rows)


In [ ]:
rows = [(0, 0)] * 30 + [(0, 1)] * 20 + [(1, 0)] * 15 + [(1, 1)] * 35
data_r = from_records(("X", "Y"), rows)
print({k: round(v, 4) for k, v in sorted(data_r.P.items())})
assert abs(data_r.P[(1, 1)] - data.P[(1, 1)]) < 1e-12


## From CSV (with or without pandas)


In [ ]:
try:
    import pandas as pd
    from partialid import from_dataframe
    data_df = from_dataframe(pd.read_csv(CSV), columns=["X", "Y"])
    print("via from_dataframe:", data_df.notes)
except ImportError:
    lines = CSV.read_text().strip().splitlines()
    header = lines[0].split(",")
    rows = [tuple(int(x) for x in line.split(",")) for line in lines[1:]]
    data_df = from_records(header, rows)
    print("via from_records (pandas not installed)")

r_csv = bound(data_df.variables, data_df.P, Q, Assumptions.sensitivity_msm("X", "Y", 3.0))
print(r_csv.summary(Q))
